In [10]:
%load_ext autoreload
%autoreload 2

from ase.io import write
from monty.serialization import loadfn
from crystring.representation import get_representation, get_crystal_structure
from crystring.deduplicate import Deduplicator
from pymatgen.io.ase import AseAtomsAdaptor
from pymatgen.analysis.structure_matcher import StructureMatcher
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [6]:
entries = loadfn("/home/qfu1/data/MP_data.json.gz")

In [7]:
entriessublist = dict(list(entries.items())[200:300])
len(entriessublist.keys())

100

In [17]:
def general_check(entries:dict):
    dif_ids_AMD = []
    dif_ids_dedup = []
    representation_dict = dict()
    crystal_from_str_dict = dict()
    ISSUE = []
    for id in entries.keys():
        entry = entries[id]
        print(id)
        
        MP_structure = AseAtomsAdaptor.get_atoms(entry["structure"])
        structure = SpacegroupAnalyzer(entry["structure"],symprec=1e-3)
        lattice = structure.get_conventional_standard_structure().lattice
        uc = {'a': lattice.a, 'b': lattice.b, 'c': lattice.c, 'alpha': lattice.alpha, 'beta': lattice.beta, 'gamma': lattice.gamma}
        try:
            
            representation, unit_cell = get_representation(MP_structure)
            representation_dict[id] = representation
            print(representation)
            
            crystal_from_str = get_crystal_structure(representation, uc)
            crystal_from_str_dict[id] = crystal_from_str
            
            # deduplicate
            dedup = Deduplicator(tol = 1e-2)
            results = dedup.deduplicate([MP_structure, crystal_from_str])
            if len(results) == 1:
                continue
            else:
                dif_ids_dedup.append(id)
            
            # AMD
            matcher = StructureMatcher()
            if not matcher.fit(AseAtomsAdaptor.get_structure(crystal_from_str), AseAtomsAdaptor.get_structure(MP_structure)):
                dif_ids_AMD.append(id)
        except:
            ISSUE.append(id)
    return dif_ids_AMD, dif_ids_dedup, ISSUE, representation_dict

In [18]:
dif_ids_AMD, dif_ids_dedup, ISSUE, representation_dict = general_check(entriessublist)

mp-1545594
P 3 1 1 S 3d 0.192 0.668 0.668 S 3d 0.24 0.416 0.374 S 3d 0.076 0.353 0.378 S 3d 0.652 0.116 0.19 S 3d 0.619 0.484 0.041 S 3d 0.738 0.52 0.364 
mp-608100
P 3 1 1 S 3d 0.3 0.722 0.756 S 3d 0.215 0.334 0.363 S 3d 0.136 0.387 0.239 S 3d 0.582 0.063 0.318 S 3d 0.555 0.49 0.025 S 3d 0.858 0.583 0.313 
mp-655141
P 3 1 1 S 3d 0.202 0.653 0.643 S 3d 0.274 0.465 0.325 S 3d 0.121 0.379 0.337 S 3d 0.65 0.062 0.194 S 3d 0.536 0.481 0.074 S 3d 0.827 0.569 0.441 
mp-1067793
C 2/c 1 1 F 8f 0.258 0.977 0.293 
mp-1077335
C 2/c 1 1 U 4e 0.000 0.821 0.000 U 8f 0.278 0.917 0.13 
mp-1087546
C 2/c 1 1 O 8f 0.642 0.644 0.105 O 8f 0.292 0.112 0.362 
mp-1179656
C 2/c 1 1 Rb 8f 0.064 0.236 0.059 Rb 8f 0.189 0.726 0.186 
mp-1180050
C 2/c 1 1 O 8f 0.567 0.666 0.151 O 8f 0.257 0.024 0.303 
mp-1199894
C 2/c 1 1 Si 8f 0.446 0.099 0.101 Si 8f 0.361 0.04 0.106 Si 8f 0.366 0.91 0.234 Si 8f 0.331 0.785 0.119 Si 8f 0.551 0.894 0.14 Si 8f 0.434 0.611 0.993 Si 8f 0.394 0.675 0.178 Si 8f 0.525 0.652 0.995 Si 8f 0

In [19]:
print("dif_ids_AMD:",len(dif_ids_AMD))
print(dif_ids_AMD)
print("dif_ids_dedup: ",len(dif_ids_dedup))
print(dif_ids_dedup)
print("Issue ids: ",ISSUE)

dif_ids_AMD: 24
['mp-604325', 'mp-19', 'mp-1179917', 'mp-567313', 'mp-1271142', 'mp-990424', 'mp-1077457', 'mp-582819', 'mp-601273', 'mp-1055932', 'mp-1184115', 'mp-1184569', 'mp-1187073', 'mp-1187739', 'mp-169', 'mp-2629196', 'mp-2739273', 'mp-611219', 'mp-973198', 'mp-975065', 'mp-975590', 'mp-1064307', 'mp-1247808', 'mp-2018526']
dif_ids_dedup:  56
['mp-1545594', 'mp-608100', 'mp-1179656', 'mp-1199894', 'mp-557031', 'mp-557376', 'mp-604325', 'mp-19', 'mp-567630', 'mp-1179917', 'mp-567313', 'mp-1120774', 'mp-1202745', 'mp-1238818', 'mp-1271142', 'mp-990424', 'mp-1077457', 'mp-582819', 'mp-601273', 'mp-1018122', 'mp-1055932', 'mp-1094057', 'mp-1183069', 'mp-1184115', 'mp-1184569', 'mp-1184764', 'mp-1186440', 'mp-1187073', 'mp-1187739', 'mp-1187769', 'mp-1228790', 'mp-160', 'mp-161', 'mp-169', 'mp-2629196', 'mp-2739273', 'mp-541848', 'mp-569007', 'mp-569304', 'mp-569416', 'mp-569517', 'mp-569567', 'mp-611219', 'mp-680372', 'mp-86', 'mp-867126', 'mp-972364', 'mp-973198', 'mp-973571', 'm

In [19]:
import pandas as pd
f = pd.DataFrame(representation_dict.items(), columns=["id", "string"])

band_gaps = []
for id in entries.keys():
    entry = entries[id]
    band_gaps.append(entry["band_gap"])

f["band_gap"] = band_gaps

csv_file_path = 'data/representation_data.csv'
f.to_csv(csv_file_path, index=False)

PermissionError: [Errno 13] Permission denied: 'representation_data.csv'

### Save structure files

In [ ]:
for id in dif_ids_AMD:
    write(f"data_to_view/{id}-pred.cif", crystal_from_str_dict[id])
    write(f"data_to_view/{id}-true.cif", AseAtomsAdaptor.get_atoms(entries[id]["structure"]))